# Regime switching

`RegimeSwitching` follows one of N **regimes** at a time. The regime follows a Markov chain that moves once per row, with transition matrix `P[i][j]` (the probability of moving from regime `i` to `j`) and an optional initial distribution.

Each regime is a source, as for `LinearCombination`: a model, `(model, "X")`, a log-return Series, a `LinearCombination`, or another `RegimeSwitching`. Every regime's source is simulated over the whole period (inactive regimes keep running), and the output's log return at each row is the active regime's.

Fantasma simulates **log returns**; `fm.to_prices` turns them into prices.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fantasma as fm

## Parameters

In [ ]:
# Sources shared by the regimes
MARKET = fm.RandomWalk(sigma=0.2, asset="MKT")
MOMENTUM = fm.StochasticDrift(sigma=0.1, drift_mean=0.05, drift_half_life=2.0, drift_sigma=0.3, asset="MOM")

# The regimes: here two linear combinations of the shared sources
CALM = fm.LinearCombination([MARKET], weights=[0.8], sigma=0.05, asset="CALM")
WILD = fm.LinearCombination([MARKET, MOMENTUM], weights=[1.5, 1.0], sigma=0.30, asset="WILD")

# Markov chain: TRANSITION_MATRIX[i][j] = probability of moving from regime i to j per row
TRANSITION_MATRIX = [
    [0.98, 0.02],
    [0.05, 0.95],
]
INITIAL = None  # a probability per regime, or None for the stationary distribution
INITIAL_PRICE = 100.0

# Simulation
START = "2024-01-01"
FREQ = "1d"
N_STEPS = 365 * 3
SEED = 42
N_PATHS = 3

## Simulate

In [ ]:
model = fm.RegimeSwitching(
    [CALM, WILD],
    transition_matrix=TRANSITION_MATRIX,
    initial=INITIAL,
    asset="STOCK",
    initial_price=INITIAL_PRICE,
)
print("stationary distribution:", np.round(model.chain.stationary_distribution(), 4))
model

In [ ]:
paths = fm.simulate(model, start=START, freq=FREQ, n_steps=N_STEPS, seed=SEED, n_paths=N_PATHS, return_hidden=True)
paths = paths if N_PATHS > 1 else [paths]  # one path is returned itself, not in a list
res = paths[0]
pd.concat([res.output, res.regimes.add_suffix(" regime")], axis=1).head()

## Prices and regimes

The output (black) follows whichever regime is active (shaded when `WILD` is). Each regime's own price path is in `res.sources`.

In [ ]:
regime = res.regimes["STOCK"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
for name, src in res.sources.items():
    px = fm.to_prices(src.output)
    ax1.plot(px.index, px[name], lw=0.8, alpha=0.6, label=f"{name} (regime source)")
px = fm.to_prices(res.output)
ax1.plot(px.index, px["STOCK"], lw=1, color="k", label="STOCK (path 0)")
ax1.fill_between(regime.index, 0, 1, where=regime.to_numpy() == 1, transform=ax1.get_xaxis_transform(), alpha=0.1, color="red", step="pre")
ax1.set_ylabel("price")
ax1.legend(fontsize=8)
ax2.step(regime.index, regime, where="pre", lw=0.8)
ax2.set_yticks(range(model.n_regimes))
ax2.set_title("regime (path 0)")
plt.show()

## Checks

Each row's output is the active regime's log return, and the observed transitions between regimes estimate the transition matrix.

In [ ]:
for r in paths:
    reg = r.regimes["STOCK"].to_numpy()
    active = np.choose(reg, [r.sources[s].output[s].to_numpy() for s in r.sources])
    assert np.array_equal(r.output["STOCK"].to_numpy(), active)
print("every row is the active regime's return")

n = model.n_regimes
counts = np.zeros((n, n))
for r in paths:
    reg = r.regimes["STOCK"].to_numpy()
    np.add.at(counts, (reg[:-1], reg[1:]), 1)
print("true:\n", np.asarray(TRANSITION_MATRIX))
print("estimated from all paths:\n", np.round(counts / counts.sum(axis=1, keepdims=True), 3))